# Detecção de Fraude em Transações de Cartão de Crédito

Neste notebook, vamos construir um pipeline completo para detecção de fraudes, desde a exploração até a explicação das predições com SHAP.

## 1. Importação das Bibliotecas

In [ ]:
# === CORREÇÃO DE COMPATIBILIDADE PANDAS >= 2.2 / IPYTHON ===
try:
    import pandas.io.formats.format as _pformat
    if not hasattr(_pformat, 'format_name'):
        def format_name(values, formatter=None, float_format=None, na_rep='NaN', digits=None, space=False, decimal='.'):
            try:
                if hasattr(values, 'name') and values.name is not None:
                    return str(values.name)
            except Exception:
                pass
            return ''
        _pformat.format_name = format_name
    
    import pandas.io.formats.html as _phtml
    if not hasattr(_phtml, 'format_name'):
        _phtml.format_name = _pformat.format_name
except Exception as _compat_err:
    print('Aviso compatibilidade:', _compat_err)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

import shap

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Carregamento dos Dados

Os dados serão carregados diretamente do Kaggle via URL, conforme sugerido na aula.

In [ ]:
# URL do dataset Credit Card Fraud Detection
url = 'https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv'

df = pd.read_csv(url)
df.head()

## 3. Exploração dos Dados

In [ ]:
print(f'Shape do dataset: {df.shape}')
print(f'\nColunas: {df.columns.tolist()}')
df.info()

In [ ]:
df.describe()

In [ ]:
# Verificação de valores nulos
df.isnull().sum().max()

### Análise do desbalanceamento

In [ ]:
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print('Contagem:')
print(class_counts)
print('\nPercentual:')
print(class_pct)

plt.figure(figsize=(8,5))
sns.countplot(data=df, x='Class')
plt.title('Distribuição das Classes')
plt.xticks([0,1], ['Não Fraude (0)', 'Fraude (1)'])
plt.show()

**Observação:** Apenas ~0,17% das transações são fraudulentas. Isso confirma o forte desbalanceamento e justifica não usar acurácia como métrica principal.

## 4. Preparação dos Dados

In [ ]:
# Cópia do dataframe
df_prep = df.copy()

# Criação de feature: log do valor para lidar com assimetria
df_prep['Log_Amount'] = np.log1p(df_prep['Amount'])

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14,5))
sns.histplot(df_prep['Amount'], bins=50, ax=axes[0])
axes[0].set_title('Distribuição de Amount')
sns.histplot(df_prep['Log_Amount'], bins=50, ax=axes[1])
axes[1].set_title('Distribuição de Log_Amount')
plt.tight_layout()
plt.show()

In [ ]:
# Separação entre features e target
X = df_prep.drop('Class', axis=1)
y = df_prep['Class']

# Divisão treino/teste com estratificação
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f'Treino: {X_train.shape}, Teste: {X_test.shape}')
print(f'\nProporção fraude treino: {y_train.mean()*100:.4f}%')
print(f'Proporção fraude teste: {y_test.mean()*100:.4f}%')

In [ ]:
# Padronização de Time, Amount e Log_Amount
scaler = StandardScaler()

cols_to_scale = ['Time', 'Amount', 'Log_Amount']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

X_train_scaled.head()

## 5. Treinamento dos Modelos

### 5.1 Regressão Logística (Baseline)

In [ ]:
lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_proba_lr = lr.predict_proba(X_test_scaled)[:, 1]

print('=== REGRESSÃO LOGÍSTICA (BASELINE) ===')
print(classification_report(y_test, y_pred_lr, target_names=['Não Fraude', 'Fraude']))
print(f'\nROC-AUC: {roc_auc_score(y_test, y_proba_lr):.4f}')
print(f'PR-AUC: {average_precision_score(y_test, y_proba_lr):.4f}')

### 5.2 Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)
y_proba_rf = rf.predict_proba(X_test_scaled)[:, 1]

print('=== RANDOM FOREST ===')
print(classification_report(y_test, y_pred_rf, target_names=['Não Fraude', 'Fraude']))
print(f'\nROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}')
print(f'PR-AUC: {average_precision_score(y_test, y_proba_rf):.4f}')

### 5.3 XGBoost

In [ ]:
# Cálculo do scale_pos_weight
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

xgb = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)
xgb.fit(X_train_scaled, y_train)

y_pred_xgb = xgb.predict(X_test_scaled)
y_proba_xgb = xgb.predict_proba(X_test_scaled)[:, 1]

print('\n=== XGBOOST ===')
print(classification_report(y_test, y_pred_xgb, target_names=['Não Fraude', 'Fraude']))
print(f'\nROC-AUC: {roc_auc_score(y_test, y_proba_xgb):.4f}')
print(f'PR-AUC: {average_precision_score(y_test, y_proba_xgb):.4f}')

## 6. Comparação dos Modelos

In [ ]:
resultados = []

for nome, y_pred, y_proba in [
    ('Regressão Logística', y_pred_lr, y_proba_lr),
    ('Random Forest', y_pred_rf, y_proba_rf),
    ('XGBoost', y_pred_xgb, y_proba_xgb)
]:
    resultados.append({
        'Modelo': nome,
        'Recall (Fraude)': recall_score(y_test, y_pred, pos_label=1),
        'Precisão (Fraude)': precision_score(y_test, y_pred, pos_label=1),
        'F1-Score (Fraude)': f1_score(y_test, y_pred, pos_label=1),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
        'PR-AUC': average_precision_score(y_test, y_proba)
    })

df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.round(4)
df_resultados

### Curvas ROC e Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16,6))

# ROC Curve
for nome, y_proba in [('Regressão Logística', y_proba_lr), ('Random Forest', y_proba_rf), ('XGBoost', y_proba_xgb)]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    axes[0].plot(fpr, tpr, label=f'{nome} (AUC={roc_auc_score(y_test,y_proba):.3f})')

axes[0].plot([0,1],[0,1],'k--')
axes[0].set_title('Curva ROC')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

# Precision-Recall Curve
for nome, y_proba in [('Regressão Logística', y_proba_lr), ('Random Forest', y_proba_rf), ('XGBoost', y_proba_xgb)]:
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    axes[1].plot(recall, precision, label=f'{nome} (AP={average_precision_score(y_test,y_proba):.3f})')

axes[1].set_title('Curva Precision-Recall')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precisão')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Ajuste do Limiar de Decisão

O limiar padrão de 0.5 nem sempre é o ideal em cenários desbalanceados. Vamos analisar o trade-off Precision-Recall e escolher um limiar mais adequado para priorizar **recall** da classe fraude.

In [ ]:
# Usando XGBoost como modelo base para ajuste de limiar (geralmente melhor performance)
y_proba_best = y_proba_xgb

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_best)

# Plot
plt.figure(figsize=(10,6))
plt.plot(thresholds, precisions[:-1], label='Precisão')
plt.plot(thresholds, recalls[:-1], label='Recall')
plt.axvline(x=0.5, color='red', linestyle='--', label='Limiar padrão (0.5)')
plt.title('Precisão vs Recall por Limiar - XGBoost')
plt.xlabel('Limiar')
plt.ylabel('Score')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Avaliação com diferentes limiares
limiares = [0.3, 0.4, 0.5, 0.6, 0.7]

for thresh in limiares:
    y_pred_thresh = (y_proba_best >= thresh).astype(int)
    print(f'\n=== LIMIAR = {thresh} ===')
    print(f'Recall (Fraude): {recall_score(y_test, y_pred_thresh, pos_label=1):.4f}')
    print(f'Precisão (Fraude): {precision_score(y_test, y_pred_thresh, pos_label=1):.4f}')
    print(f'F1-Score (Fraude): {f1_score(y_test, y_pred_thresh, pos_label=1):.4f}')
    print(confusion_matrix(y_test, y_pred_thresh))

**Escolha do limiar:** Neste caso, um limiar entre 0.3 e 0.4 permite capturar mais fraudes (maior recall) com uma perda ainda aceitável na precisão. A decisão final deve considerar o custo de falsos positivos vs falsos negativos no contexto de negócio. Para este exercício, vamos adotar **limiar = 0.4** priorizando o recall.

In [ ]:
# Aplicação do limiar escolhido
thresh_escolhido = 0.4
y_pred_final = (y_proba_best >= thresh_escolhido).astype(int)

print(f'=== XGBOOST COM LIMIAR = {thresh_escolhido} ===')
print(classification_report(y_test, y_pred_final, target_names=['Não Fraude', 'Fraude']))
print(f'\nMatriz de Confusão:')
print(confusion_matrix(y_test, y_pred_final))

## 8. Importância das Variáveis e SHAP

In [ ]:
# Feature Importance do XGBoost
feat_importance = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'importance': xgb.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10,8))
sns.barplot(data=feat_importance.head(15), x='importance', y='feature')
plt.title('Top 15 Features - XGBoost Feature Importance')
plt.tight_layout()
plt.show()

feat_importance.head(15)

### SHAP Values

In [ ]:
# Calculando SHAP values (usando amostra para performance)
explainer = shap.TreeExplainer(xgb)
X_test_sample = X_test_scaled.sample(1000, random_state=42)
shap_values = explainer.shap_values(X_test_sample)

print('SHAP values calculados com sucesso!')

In [ ]:
# Summary Plot - Importância global
shap.summary_plot(shap_values, X_test_sample, plot_type='bar', show=False)
plt.title('SHAP - Importância Global das Features')
plt.tight_layout()
plt.show()

In [ ]:
# Summary Plot detalhado
shap.summary_plot(shap_values, X_test_sample, show=False)
plt.title('SHAP - Impacto das Features nas Predições')
plt.tight_layout()
plt.show()

## 9. Conclusão

- **Desbalanceamento**: Com apenas 0,17% de fraudes, acurácia é enganosa. Métricas como Recall, Precisão e F1 são essenciais.
- **Modelos**: XGBoost com `scale_pos_weight` apresentou o melhor desempenho geral, especialmente após ajuste de limiar.
- **Limiar**: Ajustar o threshold (de 0.5 para ~0.4) permite maior captura de fraudes, sendo fundamental em problemas de detecção de anomalias.
- **Explicabilidade**: SHAP revela que componentes PCA (V14, V12, V10, etc.) são os maiores preditores, o que faz sentido dado que foram gerados para capturar padrões discriminativos.
- **Abordagem prática**: Combinar tratamento adequado do desbalanceamento + métricas corretas + ajuste de limiar + explicabilidade resulta em um modelo mais confiável para aplicação real.